# 02 — Risk Model & Threshold Analysis

## Question

Can transaction signals rank fraud risk, and what happens when the control threshold changes?

This notebook builds a baseline fraud risk model and uses its scores to simulate the trade-off between fraud capture and legitimate transaction friction.

The model is a decision-analysis tool rather than the final product. The main objective is to understand how different thresholds change who would be flagged, how much fraud would be captured, and how much legitimate activity would also be interrupted.

Because the source data does not contain actual approve, decline, review, or customer-friction outcomes, all threshold outcomes are simulated.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = Path("/content/drive/MyDrive/PROJECTS/fraud-friction")
RAW = BASE / "data" / "raw"

transactions = pd.read_csv(RAW / "train_transaction.csv")

print(f"Transactions: {len(transactions):,}")
print(f"Columns: {transactions.shape[1]}")


## 1. Build a temporal modeling split

`TransactionDT` is an anonymized time delta, so I convert it into relative days, weeks, and hours.

The model uses an ordered split rather than a random split:

- **Train:** Weeks 1–18
- **Validation:** Weeks 19–22
- **Test:** Weeks 23–26

This preserves the direction of time and makes validation more similar to scoring future transactions from earlier observations.


In [ ]:
SECONDS_PER_DAY = 86400
min_dt = transactions["TransactionDT"].min()

transactions["RelativeDay"] = (
    (transactions["TransactionDT"] - min_dt) // SECONDS_PER_DAY
).astype(int) + 1

transactions["RelativeWeek"] = (
    (transactions["RelativeDay"] - 1) // 7
).astype(int) + 1

transactions["Hour"] = (
    (transactions["TransactionDT"] % SECONDS_PER_DAY) // 3600
).astype(int)

train = transactions[transactions["RelativeWeek"] <= 18].copy()

valid = transactions[
    transactions["RelativeWeek"].between(19, 22)
].copy()

test = transactions[
    transactions["RelativeWeek"].between(23, 26)
].copy()

split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "weeks": ["1–18", "19–22", "23–26"],
    "transactions": [len(train), len(valid), len(test)],
    "fraud_rate": [
        train["isFraud"].mean(),
        valid["isFraud"].mean(),
        test["isFraud"].mean()
    ]
})

split_summary


## 2. Build an interpretable baseline risk model

The feature set combines transaction amount, relative time, card attributes, address fields, product category, email domains, and match indicators.

Numeric missing values are median-imputed. Categorical missing values are filled with the most frequent category and then one-hot encoded.

Logistic regression with balanced class weights is used as a transparent baseline for producing a continuous risk-ranking score. Because class weighting changes the fitted score distribution, the output is treated as a **model risk score**, not a calibrated probability of fraud.


In [ ]:
numeric_features = [
    "TransactionAmt",
    "RelativeDay",
    "Hour",
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1",
    "addr2"
]

categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
    "R_emaildomain",
    "M1",
    "M2",
    "M3",
    "M4",
    "M5",
    "M6",
    "M7",
    "M8",
    "M9"
]

features = numeric_features + categorical_features

X_train = train[features]
y_train = train["isFraud"]

X_valid = valid[features]
y_valid = valid["isFraud"]

X_test = test[features]
y_test = test["isFraud"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=100
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear"
    ))
])

baseline_model.fit(X_train, y_train)

print(f"Features: {len(features)}")
print(f"Train: {X_train.shape}")
print(f"Validation: {X_valid.shape}")
print(f"Test: {X_test.shape}")


In [ ]:
valid_score = baseline_model.predict_proba(X_valid)[:, 1]
test_score = baseline_model.predict_proba(X_test)[:, 1]

model_performance = pd.DataFrame({
    "split": ["Validation", "Test"],
    "ROC_AUC": [
        roc_auc_score(y_valid, valid_score),
        roc_auc_score(y_test, test_score)
    ],
    "PR_AUC": [
        average_precision_score(y_valid, valid_score),
        average_precision_score(y_test, test_score)
    ],
    "fraud_prevalence": [
        y_valid.mean(),
        y_test.mean()
    ]
})

model_performance.round(4)


### Finding 1

The model is evaluated with both ROC-AUC and PR-AUC.

PR-AUC is especially useful here because fraud is a small minority of transactions. The comparison with fraud prevalence provides context for whether the model is concentrating fraud above the underlying base rate.

The goal is not to claim a production-ready fraud model. It is to establish whether the score contains enough ranking information to support threshold analysis.


## 3. Does the risk score actually rank fraud?

Validation transactions are divided into ten equal-sized groups based on model risk score.

If the score is useful for control decisions, observed fraud should become more concentrated in the higher-risk groups.


In [ ]:
valid_scored = valid[
    [
        "TransactionID",
        "RelativeWeek",
        "ProductCD",
        "TransactionAmt",
        "isFraud"
    ]
].copy()

valid_scored["risk_score"] = valid_score

valid_scored["risk_decile"] = pd.qcut(
    valid_scored["risk_score"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

decile_summary = (
    valid_scored
    .groupby("risk_decile")
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum"),
        avg_risk=("risk_score", "mean"),
        fraud_rate=("isFraud", "mean")
    )
    .reset_index()
)

decile_summary.round(4)


In [ ]:
bottom_decile_rate = decile_summary.loc[
    decile_summary["risk_decile"] == decile_summary["risk_decile"].min(),
    "fraud_rate"
].iloc[0]

top_decile_rate = decile_summary.loc[
    decile_summary["risk_decile"] == decile_summary["risk_decile"].max(),
    "fraud_rate"
].iloc[0]

print(f"Bottom risk decile fraud rate: {bottom_decile_rate:.2%}")
print(f"Top risk decile fraud rate:    {top_decile_rate:.2%}")
print(f"Top-to-bottom concentration:   {top_decile_rate / bottom_decile_rate:.1f}x")


### Finding 2

Fraud becomes substantially more concentrated as the model risk score increases.

In the original validation run, the top risk decile had an observed fraud rate of about **14.9%**, compared with about **0.6%** in the bottom decile, a concentration of roughly **25.8×**.

This supports using the score as a ranking signal. It does not mean the score itself is a calibrated probability of fraud.


## 4. What changes when the risk threshold moves?

A transaction is treated as flagged when its model risk score is greater than or equal to the selected threshold.

For each threshold, I calculate:

- **Fraud capture rate:** share of observed fraud transactions flagged
- **False positive rate:** share of legitimate transactions flagged
- **Precision:** share of flagged transactions that are actually fraud
- **Fraud caught / missed:** transaction counts under the simulated policy
- **Legitimate interrupted:** legitimate transactions that would be flagged
- **Value captured / missed / interrupted:** transaction value associated with each outcome

`Legitimate interrupted` is a simulated policy outcome. The dataset does not show that these transactions were actually declined or that customers experienced observed friction.


In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)

amount = valid["TransactionAmt"].to_numpy()
actual = y_valid.to_numpy()

results = []

for threshold in thresholds:

    flagged = valid_score >= threshold

    fraud_mask = actual == 1
    legit_mask = actual == 0

    fraud_caught = flagged & fraud_mask
    fraud_missed = (~flagged) & fraud_mask
    legit_interrupted = flagged & legit_mask

    results.append({
        "threshold": threshold,
        "fraud_capture": fraud_caught.sum() / fraud_mask.sum(),
        "false_positive_rate": legit_interrupted.sum() / legit_mask.sum(),
        "precision": (
            fraud_caught.sum() / flagged.sum()
            if flagged.sum() else 0
        ),
        "fraud_transactions_caught": fraud_caught.sum(),
        "fraud_transactions_missed": fraud_missed.sum(),
        "legitimate_transactions_interrupted": legit_interrupted.sum(),
        "fraud_value_captured": amount[fraud_caught].sum(),
        "fraud_value_missed": amount[fraud_missed].sum(),
        "legitimate_value_interrupted": amount[legit_interrupted].sum()
    })

threshold_results = pd.DataFrame(results)

threshold_results.round(4)


In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    threshold_results["false_positive_rate"],
    threshold_results["fraud_capture"],
    marker="o"
)

for _, row in threshold_results.iterrows():
    plt.annotate(
        f"{row['threshold']:.2f}",
        (
            row["false_positive_rate"],
            row["fraud_capture"]
        ),
        fontsize=8
    )

plt.xlabel("False Positive Rate / Simulated Legitimate Friction")
plt.ylabel("Fraud Capture Rate")
plt.title("Fraud Capture vs Simulated Legitimate Transaction Friction")
plt.tight_layout()
plt.show()


### Finding 3

There is no threshold that simply "maximizes fraud detection" without a trade-off.

Lower thresholds capture more observed fraud, but they also flag more legitimate transactions. Higher thresholds reduce legitimate interruption, but more fraud is missed.

For example, in the original validation run:

| Threshold | Fraud capture | False positive rate | Precision | Fraud caught | Fraud missed | Legitimate interrupted |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0.70 | 54.0% | 14.4% | 11.6% | 1,423 | 1,214 | 10,819 |
| 0.75 | 46.3% | 10.2% | 13.7% | 1,220 | 1,417 | 7,672 |
| 0.85 | 28.5% | 3.9% | 20.4% | 751 | 1,886 | 2,939 |

The table is not used to declare an optimal threshold. A real operating threshold would depend on information this dataset does not provide, including fraud loss, review capacity, intervention cost, customer impact, and the consequences of false positives.


## Conclusion

The baseline model produces a useful fraud-risk ranking, but the more important result is what happens when that score becomes a control decision.

Moving the threshold changes fraud capture, legitimate transaction interruption, precision, and transaction value exposure at the same time. A threshold is therefore not just a model setting. It represents an operating trade-off.

The evidence supports treating fraud control as a decision system rather than evaluating the model from a single classification metric.

### What this analysis does not prove

- The model risk score is not presented as a calibrated fraud probability.
- A flagged legitimate transaction is not evidence of an actual decline or customer complaint.
- `Legitimate interrupted` is simulated friction, not observed customer experience.
- Transaction value associated with fraud is not the same as verified financial loss.
- No threshold is claimed to be optimal without business costs, review capacity, and intervention outcomes.
- Model performance on this historical dataset does not establish production performance on future transactions.
- The analysis identifies associations and ranking signals, not causal drivers of fraud.
